# 02 - Model Prototype
Verify the full model pipeline: load components, run a forward pass, check tensor shapes.

In [ ]:
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Individual Components

In [ ]:
from src.main.model.audio_encoder import AudioEncoder

audio_enc = AudioEncoder(model_name='openai/whisper-small', freeze=True).to(device)
dummy_audio = torch.randn(2, 80, 3000).to(device)
audio_out = audio_enc(dummy_audio)
print(f'AudioEncoder output: {audio_out.shape}')  # [2, 1500, 768]

In [ ]:
from src.main.model.text_encoder import TextEncoder

text_enc = TextEncoder(model_name='facebook/bart-base').to(device)
dummy_ids = torch.randint(0, 50265, (2, 64)).to(device)
dummy_mask = torch.ones(2, 64, dtype=torch.long).to(device)
text_out = text_enc(dummy_ids, dummy_mask)
print(f'TextEncoder output: {text_out.shape}')  # [2, 64, 768]

In [ ]:
from src.main.model.fusion import CrossModalFusion

fusion = CrossModalFusion(audio_dim=768, text_dim=768, fused_dim=768).to(device)
fused, fused_mask = fusion(audio_out, text_out, text_mask=dummy_mask.float())
print(f'Fusion output: {fused.shape}')       # [2, 1500+64, 768]
print(f'Fusion mask:   {fused_mask.shape}')  # [2, 1564]

## 2. Full Model Forward Pass

In [ ]:
from src.main.model.main_model import MainModel

model = MainModel(
    whispher_model='openai/whisper-small',
    bart_model='facebook/bart-base',
    freeze_audio=True
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params')

In [ ]:
# Dummy batch
batch = {
    'audio_features':     torch.randn(2, 80, 3000).to(device),
    'text_input_ids':     torch.randint(0, 50265, (2, 64)).to(device),
    'text_attention_mask': torch.ones(2, 64, dtype=torch.long).to(device),
    'labels':             torch.randint(0, 50265, (2, 16)).to(device),
}

output = model(**batch)
print(f'Loss:   {output.loss.item():.4f}')
print(f'Logits: {output.logits.shape}')  # [2, 16, 50265]

## 3. Generation (Inference)

In [ ]:
model.eval()
with torch.no_grad():
    generated = model.generate(
        audio_features=batch['audio_features'],
        text_input_ids=batch['text_input_ids'],
        text_attention_mask=batch['text_attention_mask'],
        max_length=32,
        num_beams=3
    )
print(f'Generated shape: {generated.shape}')  # [2, L_generated]

from transformers import BartTokenizer
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')
for i, ids in enumerate(generated):
    print(f'  Sample {i}: {tokenizer.decode(ids, skip_special_tokens=True)}')

## 4. Real Data Sample Through Model

In [ ]:
from torch.utils.data import DataLoader
from src.main.utils.dataset import CustomDataset
from src.main.utils.collator import Collator

manifest_path = str(PROJECT_ROOT / 'src' / 'data' / 'audio' / 'val_manifest.json')
dataset = CustomDataset(manifest_path=manifest_path)
collator = Collator()
loader = DataLoader(dataset, batch_size=2, collate_fn=collator)

real_batch = next(iter(loader))
real_batch = {k: v.to(device) for k, v in real_batch.items()}

with torch.no_grad():
    out = model(**real_batch)

print(f'Real batch loss:   {out.loss.item():.4f}')
print(f'Real batch logits: {out.logits.shape}')